# import packages

In [19]:
import numpy as np
import pandas as pd
import os

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA

from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier
from tabpfn import TabPFNClassifier

import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

from sklearn.preprocessing import OneHotEncoder

from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_curve, roc_auc_score

from sklearn.model_selection import LeaveOneGroupOut


### Load data

In [20]:
patient_data = pd.read_csv(
    '../data/0.clinical_data.csv',
)

patient_data

,Sample_ID,Age,Sex,BMI,Country,Stage,Tumor_Site,Group,Cohort,Diagnosis,Patient_ID,Age_class,Continent
0,CCMD10032470ST-11-0,45,male,30.7,Germany,NaN,NaN,Control,Wirbel_2019,Control,CCMD10032470ST-11-0,EO,Europe
1,CCMD10191450ST-11-0,62,female,28.5,Germany,NaN,NaN,Control,Wirbel_2019,Control,CCMD10191450ST-11-0,LO,Europe
2,CCMD11006829ST-21-0,42,female,35.0,Germany,II,Colon,CRC,Wirbel_2019,CRC,CCMD11006829ST-21-0,EO,Europe
3,CCMD12232071ST-21-0,75,male,28.0,Germany,III,Colon,CRC,Wirbel_2019,CRC,CCMD12232071ST-21-0,LO,Europe
4,CCMD13071240ST-21-0,66,female,23.0,Germany,II,Colon,CRC,Wirbel_2019,CRC,CCMD13071240ST-21-0,LO,Europe
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2463,cohort4_VF443,82,female,NaN,Italy,III,Colon,CRC,Piccinno_2025_IIGM_IT,CRC,cohort4_VF443,LO,Europe
2464,cohort4_VF347,51,female,24.5,Italy,IV,Colon,CRC,Piccinno_2025_IIGM_IT,CRC,cohort4_VF347,LO,Europe
2465,cohort4_VF442,86,male,25.1,Italy,Unknown_stage,Colon,CRC,Piccinno_2025_IIGM_IT,CRC,cohort4_VF442,LO,Europe
2466,cohort4_VF274,50,male,23.4,Italy,NaN,NaN,Control,Piccinno_2025_IIGM_IT,Control,cohort4_VF274,LO,Europe


In [21]:
judgement = pd.read_csv(
    '../data/0.model_cohort_determine.csv'
)

judgement

,Cohort,EOControl,LOControl,EOCRC,LOCRC,EO_all,LO_all,EO_tiny_cohort,LO_tiny_cohort,EO_test,LO_test,Cohort_true_name
0,FUSCC-SHSD,124,122,142,169,266,291,no,no,yes,yes,FUSCC-SHSD
1,Feng_2015,3,60,4,42,7,102,yes,no,no,yes,Feng_2015
2,Gupta_2019,15,15,4,26,19,41,no,no,yes,yes,Gupta_2019
3,Hannigan_2018,6,20,6,20,12,40,yes,no,no,yes,Hannigan_2018
4,Liu_2022,12,72,17,60,29,132,no,no,yes,yes,Liu_2022
5,Piccinno_2025_IIGM_CZ,9,37,4,55,13,92,yes,no,no,yes,Piccinno_2025_IIGM_CZ
6,Piccinno_2025_IIGM_IT,14,45,1,84,15,129,yes,no,no,yes,Piccinno_2025_IIGM_IT
7,Piccinno_2025_IIGM_TU,17,22,0,18,17,40,yes,no,no,yes,Piccinno_2025_IIGM_TU
8,ThomasAM_2018b,4,23,4,28,8,51,yes,no,no,yes,ThomasAM_2018b
9,Vogtmann_2016,11,41,8,44,19,85,no,no,yes,yes,Vogtmann_2016


In [22]:
for i in range(len(judgement)):
    if judgement.iloc[i,5] < 15:
        judgement.iloc[i,7] = 'yes'
        judgement.iloc[i,9] = 'no'

judgement

,Cohort,EOControl,LOControl,EOCRC,LOCRC,EO_all,LO_all,EO_tiny_cohort,LO_tiny_cohort,EO_test,LO_test,Cohort_true_name
0,FUSCC-SHSD,124,122,142,169,266,291,no,no,yes,yes,FUSCC-SHSD
1,Feng_2015,3,60,4,42,7,102,yes,no,no,yes,Feng_2015
2,Gupta_2019,15,15,4,26,19,41,no,no,yes,yes,Gupta_2019
3,Hannigan_2018,6,20,6,20,12,40,yes,no,no,yes,Hannigan_2018
4,Liu_2022,12,72,17,60,29,132,no,no,yes,yes,Liu_2022
5,Piccinno_2025_IIGM_CZ,9,37,4,55,13,92,yes,no,no,yes,Piccinno_2025_IIGM_CZ
6,Piccinno_2025_IIGM_IT,14,45,1,84,15,129,yes,no,no,yes,Piccinno_2025_IIGM_IT
7,Piccinno_2025_IIGM_TU,17,22,0,18,17,40,yes,no,no,yes,Piccinno_2025_IIGM_TU
8,ThomasAM_2018b,4,23,4,28,8,51,yes,no,no,yes,ThomasAM_2018b
9,Vogtmann_2016,11,41,8,44,19,85,no,no,yes,yes,Vogtmann_2016


In [23]:
df0 = pd.read_csv(
    '../data/input_data0.ml_input_data_raw.csv',
)

df0

,clade_name,CCMD45004878ST-11-0,CCMD85481373ST-11-0,CCMD89643949ST-11-0,CCMD45812507ST-11-0,CCMD52117727ST-11-0,CCMD54057834ST-11-0,CCMD31134579ST-11-0,CCMD27463710ST-11-0,CCMD38158721ST-11-0,...,cohort4_LILT_VF142_T017,VF208,VF273,cohort4_LILT_VF75_16,cohort4_VF402,VF107,VF265,cohort4_LILT_VF106_16,cohort4_LILT_VF167_T017,cohort4_LILT_VF360
0,k__Bacteria|p__Tenericutes|c__CFGB1787|o__OFGB...,10.04832,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,...,0.00000,0.00000,0.0,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000
1,k__Bacteria|p__Firmicutes|c__Clostridia|o__Eub...,6.52606,1.86312,1.12078,3.45798,1.08215,4.09017,0.83283,0.26052,1.15385,...,0.23292,0.00000,0.0,0.16061,2.06149,11.27324,5.15439,0.00000,0.01357,1.05572
2,k__Bacteria|p__Firmicutes|c__Clostridia|o__Eub...,4.23719,1.38439,0.00299,18.55324,4.72425,0.23425,0.19570,1.43305,7.41026,...,0.00000,0.00000,0.0,0.18839,1.84046,0.48032,1.84083,0.14956,0.00000,0.00000
3,k__Bacteria|p__Firmicutes|c__CFGB38642|o__OFGB...,2.73540,5.33181,1.67608,0.02818,0.00000,3.48310,0.14138,0.00000,0.00000,...,0.00000,0.00000,0.0,0.00000,0.00000,0.00000,1.74257,0.00000,0.78475,0.01637
4,k__Bacteria|p__Firmicutes|c__CFGB79294|o__OFGB...,2.70085,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,...,0.00000,0.00000,0.0,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3598,k__Bacteria|p__Firmicutes|c__Clostridia|o__Eub...,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,...,0.00000,0.00000,0.0,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000
3599,k__Bacteria|p__Proteobacteria|c__Betaproteobac...,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,...,0.00000,0.00000,0.0,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000
3600,k__Bacteria|p__Proteobacteria|c__Alphaproteoba...,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,...,0.00000,0.00000,0.0,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000
3601,k__Bacteria|p__Actinobacteria|c__Actinomycetia...,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,...,0.00000,0.00000,0.0,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000


In [24]:
df0 = df0.T
df0.columns = df0.iloc[0]
df0 = df0[1:]
df0 = df0.reset_index(names="Index")
df0.rename(columns={df0.columns[0]: "Sample_ID"}, inplace=True)
df0

clade_name,Sample_ID,k__Bacteria|p__Tenericutes|c__CFGB1787|o__OFGB1787|f__FGB1787|g__GGB4750|s__GGB4750_SGB6579,k__Bacteria|p__Firmicutes|c__Clostridia|o__Eubacteriales|f__Eubacteriales_incertae_sedis|g__Candidatus_Cibionibacter|s__Candidatus_Cibionibacter_quicibialis,k__Bacteria|p__Firmicutes|c__Clostridia|o__Eubacteriales|f__Lachnospiraceae|g__Lachnospiraceae_unclassified|s__Eubacterium_rectale,k__Bacteria|p__Firmicutes|c__CFGB38642|o__OFGB38642|f__FGB38642|g__GGB9758|s__GGB9758_SGB15368,k__Bacteria|p__Firmicutes|c__CFGB79294|o__OFGB79294|f__FGB79294|g__GGB9673|s__GGB9673_SGB15172,k__Bacteria|p__Firmicutes|c__Clostridia|o__Eubacteriales|f__Oscillospiraceae|g__GGB9635|s__GGB9635_SGB15106,k__Bacteria|p__Firmicutes|c__Clostridia|o__Eubacteriales|f__Oscillospiraceae|g__Faecalibacterium|s__Faecalibacterium_prausnitzii,k__Bacteria|p__Firmicutes|c__CFGB1798|o__OFGB1798|f__FGB1798|g__GGB4769|s__GGB4769_SGB6602,k__Bacteria|p__Firmicutes|c__CFGB2942|o__OFGB2942|f__FGB2942|g__GGB9278|s__GGB9278_SGB14230,...,k__Bacteria|p__Candidatus_Melainabacteria|c__CFGB2104|o__OFGB2104|f__FGB2104|g__GGB5983|s__GGB5983_SGB8603,k__Bacteria|p__Bacteroidota|c__Bacteroidia|o__Bacteroidales|f__Bacteroidaceae|g__GGB47740|s__GGB47740_SGB65712,k__Bacteria|p__Proteobacteria|c__Epsilonproteobacteria|o__Campylobacterales|f__Campylobacteraceae|g__Campylobacter|s__Campylobacter_SGB96438,k__Bacteria|p__Firmicutes|c__Clostridia|o__Eubacteriales|f__Lachnospiraceae|g__Catonella|s__Catonella_SGB4505,k__Eukaryota|p__Ascomycota|c__Saccharomycetes|o__Saccharomycetales|f__Pichiaceae|g__Pichia|s__Pichia_fermentans,k__Bacteria|p__Firmicutes|c__Clostridia|o__Eubacteriales|f__Oscillospiraceae|g__GGB74398|s__GGB74398_SGB53804,k__Bacteria|p__Proteobacteria|c__Betaproteobacteria|o__Burkholderiales|f__Comamonadaceae|g__Comamonas|s__Comamonas_testosteroni,k__Bacteria|p__Proteobacteria|c__Alphaproteobacteria|o__Caulobacterales|f__Caulobacteraceae|g__Brevundimonas|s__Brevundimonas_SGB99609,k__Bacteria|p__Actinobacteria|c__Actinomycetia|o__Bifidobacteriales|f__Bifidobacteriaceae|g__Bifidobacterium|s__Bifidobacterium_canis,k__Bacteria|p__Actinobacteria|c__Actinomycetia|o__Propionibacteriales|f__Propionibacteriaceae|g__Cutibacterium|s__Cutibacterium_namnetense
0,CCMD45004878ST-11-0,10.04832,6.52606,4.23719,2.7354,2.70085,2.43639,2.15602,2.04055,2.0115,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,CCMD85481373ST-11-0,0.0,1.86312,1.38439,5.33181,0.0,0.70455,5.97422,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,CCMD89643949ST-11-0,0.0,1.12078,0.00299,1.67608,0.0,1.31568,10.80576,0.0,0.46447,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,CCMD45812507ST-11-0,0.0,3.45798,18.55324,0.02818,0.0,0.18245,8.76529,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,CCMD52117727ST-11-0,0.0,1.08215,4.72425,0.0,0.0,0.03428,3.37867,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2463,VF107,0.0,11.27324,0.48032,0.0,0.0,0.0,4.4675,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2464,VF265,0.0,5.15439,1.84083,1.74257,0.0,0.35792,7.2697,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2465,cohort4_LILT_VF106_16,0.0,0.0,0.14956,0.0,0.0,0.0,20.3692,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2466,cohort4_LILT_VF167_T017,0.0,0.01357,0.0,0.78475,0.0,0.0,1.49187,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [25]:
aaa = df0.drop(columns=["Sample_ID"])
global_min = aaa.replace(0, np.inf).to_numpy().min()

print(f"min value (excluding zeros): {global_min}")

min value (excluding zeros): 1e-05


/tmp/ipykernel_136090/1987367766.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  global_min = aaa.replace(0, np.inf).to_numpy().min()


In [26]:
df = df0.drop(columns=["Sample_ID"])

df = (df.apply(pd.to_numeric, errors="coerce")
                     .replace([np.inf, -np.inf], np.nan)
                     .fillna(0.0)
                     .astype(np.float64))

row_sums = df.sum(axis=1)
X_closed = df.div(row_sums.replace(0, np.nan), axis=0).fillna(0.0)

eps = 1e-6
X_pc = X_closed + eps

logX = np.log(X_pc)
clr = logX.sub(logX.mean(axis=1), axis=0)

df = clr

df = pd.DataFrame(df, index=df0.index, columns=df0.columns)
df["Sample_ID"] = df0["Sample_ID"]

df

clade_name,Sample_ID,k__Bacteria|p__Tenericutes|c__CFGB1787|o__OFGB1787|f__FGB1787|g__GGB4750|s__GGB4750_SGB6579,k__Bacteria|p__Firmicutes|c__Clostridia|o__Eubacteriales|f__Eubacteriales_incertae_sedis|g__Candidatus_Cibionibacter|s__Candidatus_Cibionibacter_quicibialis,k__Bacteria|p__Firmicutes|c__Clostridia|o__Eubacteriales|f__Lachnospiraceae|g__Lachnospiraceae_unclassified|s__Eubacterium_rectale,k__Bacteria|p__Firmicutes|c__CFGB38642|o__OFGB38642|f__FGB38642|g__GGB9758|s__GGB9758_SGB15368,k__Bacteria|p__Firmicutes|c__CFGB79294|o__OFGB79294|f__FGB79294|g__GGB9673|s__GGB9673_SGB15172,k__Bacteria|p__Firmicutes|c__Clostridia|o__Eubacteriales|f__Oscillospiraceae|g__GGB9635|s__GGB9635_SGB15106,k__Bacteria|p__Firmicutes|c__Clostridia|o__Eubacteriales|f__Oscillospiraceae|g__Faecalibacterium|s__Faecalibacterium_prausnitzii,k__Bacteria|p__Firmicutes|c__CFGB1798|o__OFGB1798|f__FGB1798|g__GGB4769|s__GGB4769_SGB6602,k__Bacteria|p__Firmicutes|c__CFGB2942|o__OFGB2942|f__FGB2942|g__GGB9278|s__GGB9278_SGB14230,...,k__Bacteria|p__Candidatus_Melainabacteria|c__CFGB2104|o__OFGB2104|f__FGB2104|g__GGB5983|s__GGB5983_SGB8603,k__Bacteria|p__Bacteroidota|c__Bacteroidia|o__Bacteroidales|f__Bacteroidaceae|g__GGB47740|s__GGB47740_SGB65712,k__Bacteria|p__Proteobacteria|c__Epsilonproteobacteria|o__Campylobacterales|f__Campylobacteraceae|g__Campylobacter|s__Campylobacter_SGB96438,k__Bacteria|p__Firmicutes|c__Clostridia|o__Eubacteriales|f__Lachnospiraceae|g__Catonella|s__Catonella_SGB4505,k__Eukaryota|p__Ascomycota|c__Saccharomycetes|o__Saccharomycetales|f__Pichiaceae|g__Pichia|s__Pichia_fermentans,k__Bacteria|p__Firmicutes|c__Clostridia|o__Eubacteriales|f__Oscillospiraceae|g__GGB74398|s__GGB74398_SGB53804,k__Bacteria|p__Proteobacteria|c__Betaproteobacteria|o__Burkholderiales|f__Comamonadaceae|g__Comamonas|s__Comamonas_testosteroni,k__Bacteria|p__Proteobacteria|c__Alphaproteobacteria|o__Caulobacterales|f__Caulobacteraceae|g__Brevundimonas|s__Brevundimonas_SGB99609,k__Bacteria|p__Actinobacteria|c__Actinomycetia|o__Bifidobacteriales|f__Bifidobacteriaceae|g__Bifidobacterium|s__Bifidobacterium_canis,k__Bacteria|p__Actinobacteria|c__Actinomycetia|o__Propionibacteriales|f__Propionibacteriaceae|g__Cutibacterium|s__Cutibacterium_namnetense
0,CCMD45004878ST-11-0,10.894061,10.462465,10.030570,9.592960,9.580250,9.477204,9.354956,9.299914,9.285576,...,-0.623695,-0.623695,-0.623695,-0.623695,-0.623695,-0.623695,-0.623695,-0.623695,-0.623695,-0.623695
1,CCMD85481373ST-11-0,-0.436230,9.396416,9.099442,10.447819,-0.436230,8.424056,10.561580,-0.436230,-0.436230,...,-0.436230,-0.436230,-0.436230,-0.436230,-0.436230,-0.436230,-0.436230,-0.436230,-0.436230,-0.436230
2,CCMD89643949ST-11-0,-0.649219,8.675236,2.781538,9.077639,-0.649219,8.835552,10.941211,-0.649219,7.794479,...,-0.649219,-0.649219,-0.649219,-0.649219,-0.649219,-0.649219,-0.649219,-0.649219,-0.649219,-0.649219
3,CCMD45812507ST-11-0,-0.415122,10.035931,11.715868,5.229617,-0.415122,7.094487,10.966029,-0.415122,-0.415122,...,-0.415122,-0.415122,-0.415122,-0.415122,-0.415122,-0.415122,-0.415122,-0.415122,-0.415122,-0.415122
4,CCMD52117727ST-11-0,-0.419314,8.870069,10.343757,-0.419314,-0.419314,5.420746,10.008538,-0.419314,-0.419314,...,-0.419314,-0.419314,-0.419314,-0.419314,-0.419314,-0.419314,-0.419314,-0.419314,-0.419314,-0.419314
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2463,VF107,-0.284965,11.347816,8.192281,-0.284965,-0.284965,-0.284965,10.422227,-0.284965,-0.284965,...,-0.284965,-0.284965,-0.284965,-0.284965,-0.284965,-0.284965,-0.284965,-0.284965,-0.284965,-0.284965
2464,VF265,-0.344663,10.505545,9.475948,9.421095,-0.344663,7.838510,10.849406,-0.344663,-0.344663,...,-0.344663,-0.344663,-0.344663,-0.344663,-0.344663,-0.344663,-0.344663,-0.344663,-0.344663,-0.344663
2465,cohort4_LILT_VF106_16,-0.115588,-0.115588,7.195363,-0.115588,-0.115588,-0.115588,12.108781,-0.115588,-0.115588,...,-0.115588,-0.115588,-0.115588,-0.115588,-0.115588,-0.115588,-0.115588,-0.115588,-0

In [27]:
first_dir_name = "../data/"
os.listdir(first_dir_name)

['LODO_Meta_Sig_LO',
 '0.clinical_data.csv',
 'input_data0.ml_input_data_raw.csv',
 '0.model_cohort_determine.csv',
 'LODO_Meta_Sig_EO']

### Meta_Sig_EO

In [28]:
second_dir_name = "LODO_Meta_Sig_EO/"
full_dir_name = first_dir_name + second_dir_name
MEO_files = os.listdir(full_dir_name)
pre = full_dir_name

for post in MEO_files:
    test_data_name = post[13:-4]
    print(test_data_name)

Wirbel_2019
YachidaS_2019
ThomasAM_2018b
Piccinno_2025_IIGM_CZ
Liu_2022
Vogtmann_2016
Gupta_2019
Hannigan_2018
FUSCC-SHSD
Feng_2015
Zeller_2014
Yang_2020


In [ ]:
All_record_tab = {}

for post in MEO_files:
    test_data_name = post[13:-4]
    if judgement.loc[judgement["Cohort"] == test_data_name, "EO_test"].iloc[0] == 'no':
        All_record_tab[test_data_name] = 'NaN'
        continue
    address = pre + post
    print(address)
    MEO_feature = pd.read_csv(address)
    selected_columns = ["Sample_ID"] + MEO_feature["Feature"].tolist()
    df_MEO_feature = df[selected_columns].copy()
    patient_data_MEO = patient_data[patient_data["Age_class"].eq("EO")]
    target_MEO = patient_data_MEO[["Sample_ID", "Group", "Cohort"]].copy()
    target_MEO.rename(columns={"Group": "labels"}, inplace=True)
    df_MEO_feature["Sample_ID"] = df_MEO_feature["Sample_ID"].astype(str)
    target_MEO["Sample_ID"] = target_MEO["Sample_ID"].astype(str)
    merged_MEO_feature = pd.merge(df_MEO_feature, target_MEO, left_on="Sample_ID", right_on="Sample_ID", how="right")

    X_full = merged_MEO_feature.drop(columns=["Sample_ID", "labels", "Cohort"])
    y_full = merged_MEO_feature["labels"]
    Groups = merged_MEO_feature["Cohort"]
    
    J = Groups == test_data_name
    X_train = X_full[~J]
    y_train = y_full[~J]
    X_test = X_full[J]
    y_test = y_full[J]


    tc = TabPFNClassifier(model_path="...", # The location of the model weights on your computer
                          device="cuda") 
    tc.fit(X_train, y_train)
    proba = tc.predict_proba(X_test)
    auc = roc_auc_score((y_test == tc.classes_[1]).astype(int), proba[:, 1])
    print("ROC-AUC:", auc)
    All_record_tab[test_data_name] = auc


../data/LODO_Meta_Sig_EO/MetaRes_Test_Wirbel_2019.csv
ROC-AUC: 0.8333333333333334
../data/LODO_Meta_Sig_EO/MetaRes_Test_YachidaS_2019.csv
ROC-AUC: 0.7530637254901961
../data/LODO_Meta_Sig_EO/MetaRes_Test_Liu_2022.csv
ROC-AUC: 0.8872549019607843
../data/LODO_Meta_Sig_EO/MetaRes_Test_Vogtmann_2016.csv
ROC-AUC: 0.9772727272727273
../data/LODO_Meta_Sig_EO/MetaRes_Test_Gupta_2019.csv
ROC-AUC: 0.8166666666666667
../data/LODO_Meta_Sig_EO/MetaRes_Test_FUSCC-SHSD.csv
ROC-AUC: 0.8317810086324399
../data/LODO_Meta_Sig_EO/MetaRes_Test_Yang_2020.csv
ROC-AUC: 0.9023041474654377


In [30]:
for key in All_record_tab:
    print(f"{All_record_tab[key]}")

0.8333333333333334
0.7530637254901961
NaN
NaN
0.8872549019607843
0.9772727272727273
0.8166666666666667
NaN
0.8317810086324399
NaN
NaN
0.9023041474654377


In [31]:
All_record_rf = {}

for post in MEO_files:
    test_data_name = post[13:-4]
    if judgement.loc[judgement["Cohort"] == test_data_name, "EO_test"].iloc[0] == 'no':
        All_record_rf[test_data_name] = 'NaN'
        continue
    address = pre + post
    print(address)
    MEO_feature = pd.read_csv(address)
    selected_columns = ["Sample_ID"] + MEO_feature["Feature"].tolist()
    df_MEO_feature = df[selected_columns].copy()
    patient_data_MEO = patient_data[patient_data["Age_class"].eq("EO")]
    target_MEO = patient_data_MEO[["Sample_ID", "Group", "Cohort"]].copy()
    target_MEO.rename(columns={"Group": "labels"}, inplace=True)
    df_MEO_feature["Sample_ID"] = df_MEO_feature["Sample_ID"].astype(str)
    target_MEO["Sample_ID"] = target_MEO["Sample_ID"].astype(str)
    merged_MEO_feature = pd.merge(df_MEO_feature, target_MEO, left_on="Sample_ID", right_on="Sample_ID", how="right")
    
    X_full = merged_MEO_feature.drop(columns=["Sample_ID", "labels", "Cohort"])
    y_full = merged_MEO_feature["labels"]
    Groups = merged_MEO_feature["Cohort"]
    
    J = Groups == test_data_name
    X_train = X_full[~J]
    y_train = y_full[~J]
    X_test = X_full[J]
    y_test = y_full[J]

    rfc = RandomForestClassifier(n_estimators=500, random_state=42)
    rfc.fit(X_train, y_train)
    proba = rfc.predict_proba(X_test)
    auc = roc_auc_score((y_test == rfc.classes_[1]).astype(int), proba[:, 1])
    print("ROC-AUC:", auc)
    All_record_rf[test_data_name] = auc


../data/LODO_Meta_Sig_EO/MetaRes_Test_Wirbel_2019.csv
ROC-AUC: 0.8214285714285714
../data/LODO_Meta_Sig_EO/MetaRes_Test_YachidaS_2019.csv
ROC-AUC: 0.7365196078431372
../data/LODO_Meta_Sig_EO/MetaRes_Test_Liu_2022.csv
ROC-AUC: 0.875
../data/LODO_Meta_Sig_EO/MetaRes_Test_Vogtmann_2016.csv
ROC-AUC: 0.9886363636363636
../data/LODO_Meta_Sig_EO/MetaRes_Test_Gupta_2019.csv
ROC-AUC: 0.7416666666666667
../data/LODO_Meta_Sig_EO/MetaRes_Test_FUSCC-SHSD.csv
ROC-AUC: 0.808382553384825
../data/LODO_Meta_Sig_EO/MetaRes_Test_Yang_2020.csv
ROC-AUC: 0.9211981566820276


In [32]:
for key in All_record_rf:
    print(f"{All_record_rf[key]}")

0.8214285714285714
0.7365196078431372
NaN
NaN
0.875
0.9886363636363636
0.7416666666666667
NaN
0.808382553384825
NaN
NaN
0.9211981566820276


In [33]:
All_record_cb = {}

for post in MEO_files:
    test_data_name = post[13:-4]
    if judgement.loc[judgement["Cohort"] == test_data_name, "EO_test"].iloc[0] == 'no':
        All_record_cb[test_data_name] = 'NaN'
        continue
    address = pre + post
    print(address)
    MEO_feature = pd.read_csv(address)
    selected_columns = ["Sample_ID"] + MEO_feature["Feature"].tolist()
    df_MEO_feature = df[selected_columns].copy()
    patient_data_MEO = patient_data[patient_data["Age_class"].eq("EO")]
    target_MEO = patient_data_MEO[["Sample_ID", "Group", "Cohort"]].copy()
    target_MEO.rename(columns={"Group": "labels"}, inplace=True)
    df_MEO_feature["Sample_ID"] = df_MEO_feature["Sample_ID"].astype(str)
    target_MEO["Sample_ID"] = target_MEO["Sample_ID"].astype(str)
    merged_MEO_feature = pd.merge(df_MEO_feature, target_MEO, left_on="Sample_ID", right_on="Sample_ID", how="right")
    
    X_full = merged_MEO_feature.drop(columns=["Sample_ID", "labels", "Cohort"])
    y_full = merged_MEO_feature["labels"]
    Groups = merged_MEO_feature["Cohort"]
    
    J = Groups == test_data_name
    X_train = X_full[~J]
    y_train = y_full[~J]
    X_test = X_full[J]
    y_test = y_full[J]

    cbc = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6, verbose=False)
    cbc.fit(X_train, y_train)
    proba = cbc.predict_proba(X_test)
    auc = roc_auc_score((y_test == cbc.classes_[1]).astype(int), proba[:, 1])
    print("ROC-AUC:", auc)
    All_record_cb[test_data_name] = auc


../data/LODO_Meta_Sig_EO/MetaRes_Test_Wirbel_2019.csv
ROC-AUC: 0.7857142857142858
../data/LODO_Meta_Sig_EO/MetaRes_Test_YachidaS_2019.csv
ROC-AUC: 0.7267156862745098
../data/LODO_Meta_Sig_EO/MetaRes_Test_Liu_2022.csv
ROC-AUC: 0.8480392156862745
../data/LODO_Meta_Sig_EO/MetaRes_Test_Vogtmann_2016.csv
ROC-AUC: 0.9431818181818181
../data/LODO_Meta_Sig_EO/MetaRes_Test_Gupta_2019.csv
ROC-AUC: 0.7666666666666667
../data/LODO_Meta_Sig_EO/MetaRes_Test_FUSCC-SHSD.csv
ROC-AUC: 0.8069627442071786
../data/LODO_Meta_Sig_EO/MetaRes_Test_Yang_2020.csv
ROC-AUC: 0.9055299539170507


In [34]:
for key in All_record_cb:
    print(f"{All_record_cb[key]}")

0.7857142857142858
0.7267156862745098
NaN
NaN
0.8480392156862745
0.9431818181818181
0.7666666666666667
NaN
0.8069627442071786
NaN
NaN
0.9055299539170507


### Meta_Sig_LO

In [35]:
second_dir_name = "LODO_Meta_Sig_LO/"
full_dir_name = first_dir_name + second_dir_name
MLO_files = os.listdir(full_dir_name)
pre = full_dir_name

for post in MLO_files:
    test_data_name = post[13:-4]
    print(test_data_name)

Wirbel_2019
YachidaS_2019
ThomasAM_2018b
Piccinno_2025_IIGM_CZ
Liu_2022
Vogtmann_2016
Gupta_2019
Hannigan_2018
FUSCC-SHSD
Yu_2017
Piccinno_2025_IIGM_IT
Feng_2015
Zeller_2014
Yang_2020
Piccinno_2025_IIGM_TU


In [ ]:
All_record_tab = {}

for post in MLO_files:
    test_data_name = post[13:-4]
    if judgement.loc[judgement["Cohort"] == test_data_name, "LO_test"].iloc[0] == 'no':
        All_record_tab[test_data_name] = 'NaN'
        continue
    address = pre + post
    print(address)
    MLO_feature = pd.read_csv(address)
    selected_columns = ["Sample_ID"] + MLO_feature["Feature"].tolist()
    df_MLO_feature = df[selected_columns].copy()
    patient_data_MLO = patient_data[patient_data["Age_class"].eq("LO")]
    target_MLO = patient_data_MLO[["Sample_ID", "Group", "Cohort"]].copy()
    target_MLO.rename(columns={"Group": "labels"}, inplace=True)
    df_MLO_feature["Sample_ID"] = df_MLO_feature["Sample_ID"].astype(str)
    target_MLO["Sample_ID"] = target_MLO["Sample_ID"].astype(str)
    merged_MLO_feature = pd.merge(df_MLO_feature, target_MLO, left_on="Sample_ID", right_on="Sample_ID", how="right")
    
    X_full = merged_MLO_feature.drop(columns=["Sample_ID", "labels", "Cohort"])
    y_full = merged_MLO_feature["labels"]
    Groups = merged_MLO_feature["Cohort"]
    
    J = Groups == test_data_name
    X_train = X_full[~J]
    y_train = y_full[~J]
    X_test = X_full[J]
    y_test = y_full[J]

    tc = TabPFNClassifier(model_path="...", # The location of the model weights on your computer
                          device="cuda") 
    tc.fit(X_train, y_train)
    proba = tc.predict_proba(X_test)
    auc = roc_auc_score((y_test == tc.classes_[1]).astype(int), proba[:, 1])
    print("ROC-AUC:", auc)
    All_record_tab[test_data_name] = auc


../data/LODO_Meta_Sig_LO/MetaRes_Test_Wirbel_2019.csv
ROC-AUC: 0.9272798742138365
../data/LODO_Meta_Sig_LO/MetaRes_Test_YachidaS_2019.csv
ROC-AUC: 0.7124449682038512
../data/LODO_Meta_Sig_LO/MetaRes_Test_ThomasAM_2018b.csv
ROC-AUC: 0.8594720496894409
../data/LODO_Meta_Sig_LO/MetaRes_Test_Piccinno_2025_IIGM_CZ.csv
ROC-AUC: 0.7533169533169534
../data/LODO_Meta_Sig_LO/MetaRes_Test_Liu_2022.csv
ROC-AUC: 0.8601851851851852
../data/LODO_Meta_Sig_LO/MetaRes_Test_Vogtmann_2016.csv
ROC-AUC: 0.8209534368070953
../data/LODO_Meta_Sig_LO/MetaRes_Test_Gupta_2019.csv
ROC-AUC: 0.8256410256410257
../data/LODO_Meta_Sig_LO/MetaRes_Test_Hannigan_2018.csv
ROC-AUC: 0.6075
../data/LODO_Meta_Sig_LO/MetaRes_Test_FUSCC-SHSD.csv
ROC-AUC: 0.8697012319332622
../data/LODO_Meta_Sig_LO/MetaRes_Test_Yu_2017.csv
ROC-AUC: 0.8896825396825397
../data/LODO_Meta_Sig_LO/MetaRes_Test_Piccinno_2025_IIGM_IT.csv
ROC-AUC: 0.7197089947089947
../data/LODO_Meta_Sig_LO/MetaRes_Test_Feng_2015.csv
ROC-AUC: 0.8488095238095238
../data/LO

In [37]:
for key in All_record_tab:
    print(f"{All_record_tab[key]}")

0.9272798742138365
0.7124449682038512
0.8594720496894409
0.7533169533169534
0.8601851851851852
0.8209534368070953
0.8256410256410257
0.6075
0.8697012319332622
0.8896825396825397
0.7197089947089947
0.8488095238095238
0.8467462456680785
0.8831927319922128
0.9520202020202021


In [38]:
All_record_rf = {}

for post in MLO_files:
    test_data_name = post[13:-4]
    if judgement.loc[judgement["Cohort"] == test_data_name, "LO_test"].iloc[0] == 'no':
        All_record_rf[test_data_name] = 'NaN'
        continue
    address = pre + post
    print(address)
    MLO_feature = pd.read_csv(address)
    selected_columns = ["Sample_ID"] + MLO_feature["Feature"].tolist()
    df_MLO_feature = df[selected_columns].copy()
    patient_data_MLO = patient_data[patient_data["Age_class"].eq("LO")]
    target_MLO = patient_data_MLO[["Sample_ID", "Group", "Cohort"]].copy()
    target_MLO.rename(columns={"Group": "labels"}, inplace=True)
    df_MLO_feature["Sample_ID"] = df_MLO_feature["Sample_ID"].astype(str)
    target_MLO["Sample_ID"] = target_MLO["Sample_ID"].astype(str)
    merged_MLO_feature = pd.merge(df_MLO_feature, target_MLO, left_on="Sample_ID", right_on="Sample_ID", how="right")
    
    X_full = merged_MLO_feature.drop(columns=["Sample_ID", "labels", "Cohort"])
    y_full = merged_MLO_feature["labels"]
    Groups = merged_MLO_feature["Cohort"]
    
    J = Groups == test_data_name
    X_train = X_full[~J]
    y_train = y_full[~J]
    X_test = X_full[J]
    y_test = y_full[J]

    rfc = RandomForestClassifier(n_estimators=500, random_state=42)
    rfc.fit(X_train, y_train)
    proba = rfc.predict_proba(X_test)
    auc = roc_auc_score((y_test == rfc.classes_[1]).astype(int), proba[:, 1])
    print("ROC-AUC:", auc)
    All_record_rf[test_data_name] = auc


../data/LODO_Meta_Sig_LO/MetaRes_Test_Wirbel_2019.csv
ROC-AUC: 0.8799135220125786
../data/LODO_Meta_Sig_LO/MetaRes_Test_YachidaS_2019.csv
ROC-AUC: 0.7032507671098857
../data/LODO_Meta_Sig_LO/MetaRes_Test_ThomasAM_2018b.csv
ROC-AUC: 0.7903726708074534
../data/LODO_Meta_Sig_LO/MetaRes_Test_Piccinno_2025_IIGM_CZ.csv
ROC-AUC: 0.7169533169533169
../data/LODO_Meta_Sig_LO/MetaRes_Test_Liu_2022.csv
ROC-AUC: 0.8820601851851851
../data/LODO_Meta_Sig_LO/MetaRes_Test_Vogtmann_2016.csv
ROC-AUC: 0.7389135254988914
../data/LODO_Meta_Sig_LO/MetaRes_Test_Gupta_2019.csv
ROC-AUC: 0.8179487179487179
../data/LODO_Meta_Sig_LO/MetaRes_Test_Hannigan_2018.csv
ROC-AUC: 0.53
../data/LODO_Meta_Sig_LO/MetaRes_Test_FUSCC-SHSD.csv
ROC-AUC: 0.8459113396061693
../data/LODO_Meta_Sig_LO/MetaRes_Test_Yu_2017.csv
ROC-AUC: 0.8993386243386244
../data/LODO_Meta_Sig_LO/MetaRes_Test_Piccinno_2025_IIGM_IT.csv
ROC-AUC: 0.6788359788359788
../data/LODO_Meta_Sig_LO/MetaRes_Test_Feng_2015.csv
ROC-AUC: 0.8593253968253969
../data/LODO

In [39]:
for key in All_record_rf:
    print(f"{All_record_rf[key]}")

0.8799135220125786
0.7032507671098857
0.7903726708074534
0.7169533169533169
0.8820601851851851
0.7389135254988914
0.8179487179487179
0.53
0.8459113396061693
0.8993386243386244
0.6788359788359788
0.8593253968253969
0.8611859838274932
0.9250486696950032
0.9280303030303031


In [40]:
All_record_cb = {}

for post in MLO_files:
    test_data_name = post[13:-4]
    if judgement.loc[judgement["Cohort"] == test_data_name, "LO_test"].iloc[0] == 'no':
        All_record_cb[test_data_name] = 'NaN'
        continue
    address = pre + post
    print(address)
    MLO_feature = pd.read_csv(address)
    selected_columns = ["Sample_ID"] + MLO_feature["Feature"].tolist()
    df_MLO_feature = df[selected_columns].copy()
    patient_data_MLO = patient_data[patient_data["Age_class"].eq("LO")]
    target_MLO = patient_data_MLO[["Sample_ID", "Group", "Cohort"]].copy()
    target_MLO.rename(columns={"Group": "labels"}, inplace=True)
    df_MLO_feature["Sample_ID"] = df_MLO_feature["Sample_ID"].astype(str)
    target_MLO["Sample_ID"] = target_MLO["Sample_ID"].astype(str)
    merged_MLO_feature = pd.merge(df_MLO_feature, target_MLO, left_on="Sample_ID", right_on="Sample_ID", how="right")

    X_full = merged_MLO_feature.drop(columns=["Sample_ID", "labels", "Cohort"])
    y_full = merged_MLO_feature["labels"]
    Groups = merged_MLO_feature["Cohort"]
    
    J = Groups == test_data_name
    X_train = X_full[~J]
    y_train = y_full[~J]
    X_test = X_full[J]
    y_test = y_full[J]

    cbc = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6, verbose=False)
    cbc.fit(X_train, y_train)
    proba = cbc.predict_proba(X_test)
    auc = roc_auc_score((y_test == cbc.classes_[1]).astype(int), proba[:, 1])
    print("ROC-AUC:", auc)
    All_record_cb[test_data_name] = auc


../data/LODO_Meta_Sig_LO/MetaRes_Test_Wirbel_2019.csv
ROC-AUC: 0.914308176100629
../data/LODO_Meta_Sig_LO/MetaRes_Test_YachidaS_2019.csv
ROC-AUC: 0.7149464134833459
../data/LODO_Meta_Sig_LO/MetaRes_Test_ThomasAM_2018b.csv
ROC-AUC: 0.7981366459627329
../data/LODO_Meta_Sig_LO/MetaRes_Test_Piccinno_2025_IIGM_CZ.csv
ROC-AUC: 0.7267813267813268
../data/LODO_Meta_Sig_LO/MetaRes_Test_Liu_2022.csv
ROC-AUC: 0.8953703703703704
../data/LODO_Meta_Sig_LO/MetaRes_Test_Vogtmann_2016.csv
ROC-AUC: 0.7644124168514412
../data/LODO_Meta_Sig_LO/MetaRes_Test_Gupta_2019.csv
ROC-AUC: 0.8025641025641025
../data/LODO_Meta_Sig_LO/MetaRes_Test_Hannigan_2018.csv
ROC-AUC: 0.6525000000000001
../data/LODO_Meta_Sig_LO/MetaRes_Test_FUSCC-SHSD.csv
ROC-AUC: 0.8615772625860897
../data/LODO_Meta_Sig_LO/MetaRes_Test_Yu_2017.csv
ROC-AUC: 0.8912698412698412
../data/LODO_Meta_Sig_LO/MetaRes_Test_Piccinno_2025_IIGM_IT.csv
ROC-AUC: 0.7341269841269842
../data/LODO_Meta_Sig_LO/MetaRes_Test_Feng_2015.csv
ROC-AUC: 0.8817460317460317

In [41]:
for key in All_record_cb:
    print(f"{All_record_cb[key]}")

0.914308176100629
0.7149464134833459
0.7981366459627329
0.7267813267813268
0.8953703703703704
0.7644124168514412
0.8025641025641025
0.6525000000000001
0.8615772625860897
0.8912698412698412
0.7341269841269842
0.8817460317460317
0.8463611859838275
0.881245944192083
0.9444444444444444
